모든 케이스(검증, baseline, forest)를 엑셀데이터(particleTarget, gridTarget)만 가져오기

In [ ]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pandas as pd
import os
import re
import matplotlib.pyplot as plt

%matplotlib inline




In [4]:
save_folder = './Test_results'


In [5]:
import pandas as pd
import os
import re

def merge_targets_in_folder(folder_path, save_excel_path):
    """
    gridTarget/particleTarget 최대값을 시간축 기준으로 병합하여 저장.
    """
    all_files = sorted([
        f for f in os.listdir(folder_path)
        if (f.startswith('gridTarget') or f.startswith('particleTarget')) and f.endswith('.csv')
    ])
    if not all_files:
        print(f"[SKIP] {folder_path}: gridTarget/particleTarget csv 파일 없음")
        return

    grid_pattern = re.compile(r'(gridTarget\[\d+\])_dev\[\d+\]\.csv')
    particle_pattern = re.compile(r'(particleTarget\[\d+\])_model\[0\]_dev\[\d+\]\.csv')

    grouped_files = dict()
    particle_grouped_files = dict()
    for file in all_files:
        m = grid_pattern.match(file)
        if m:
            grid_name = m.group(1)
            grouped_files.setdefault(grid_name, []).append(file)
            continue
        p = particle_pattern.match(file)
        if p:
            particle_name = p.group(1)
            particle_grouped_files.setdefault(particle_name, []).append(file)

    if not grouped_files:
        print(f"[SKIP] {folder_path}: gridTarget 파일 그룹이 없음")
        return

    first_time_col = None
    final_dfs = []
    # gridTarget 그룹별
    for grid_name, file_list in sorted(grouped_files.items(), key=lambda x: int(re.findall(r'\d+', x[0])[0])):
        sub_dfs = []
        for idx, file in enumerate(sorted(file_list)):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path, usecols=['Time [s]', 'Force [n]'])
            time_col = 'Time [s]'
            force_col = 'Force [n]'
            if first_time_col is None:
                first_time_col = time_col
            sub_dfs.append(df)
        group_df = pd.concat(sub_dfs, axis=0, ignore_index=True)
        max_by_time = group_df.groupby(time_col, as_index=False)[force_col].max()
        max_by_time = max_by_time.rename(columns={force_col: grid_name})
        max_by_time[grid_name] = max_by_time[grid_name].round(6)
        final_dfs.append(max_by_time)
    # particleTarget 그룹별
    for particle_name, file_list in sorted(particle_grouped_files.items(), key=lambda x: int(re.findall(r'\d+', x[0])[0])):
        sub_dfs = []
        for idx, file in enumerate(sorted(file_list)):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path, usecols=['Time', 'Aggregate'])
            time_col = 'Time'
            agg_col = 'Aggregate'
            sub_dfs.append(df)
        group_df = pd.concat(sub_dfs, axis=0, ignore_index=True)
        max_by_time = group_df.groupby(time_col, as_index=False)[agg_col].max()
        max_by_time = max_by_time.rename(columns={agg_col: particle_name})
        max_by_time[particle_name] = max_by_time[particle_name].round(6)
        final_dfs.append(max_by_time)

    if len(final_dfs) == 0:
        print(f"[SKIP] {folder_path}: 최종 생성된 데이터프레임 없음")
        return

    # 시간축 맞춤
    for idx in range(len(final_dfs)):
        df = final_dfs[idx]
        if first_time_col not in df.columns and 'Time' in df.columns:
            df = df.rename(columns={'Time': first_time_col})
            df[first_time_col] = df[first_time_col].astype(float)
            final_dfs[idx] = df

    # outer merge
    def memory_efficient_outer_merge(dfs, time_col):
        result = dfs[0]
        for df in dfs[1:]:
            result = pd.merge(result, df, on=time_col, how='outer')
        return result
    merged_final = memory_efficient_outer_merge(final_dfs, first_time_col)
    for col in merged_final.columns:
        if col != first_time_col:
            merged_final[col] = merged_final[col].round(6)
    # 저장
    merged_final.to_excel(save_excel_path, index=False, float_format='%.6f')
    print(f"[DONE] {folder_path} → {save_excel_path}")


Forest case에 적용

In [6]:
import os

print("하위폴더 목록:")
forest_base = '../Projects/OSU_LWF/DigitalTwin/Test/'
os.makedirs(save_folder, exist_ok=True)

for d in sorted(os.listdir(forest_base)):
    full_path = os.path.join(forest_base, d)
    if os.path.isdir(full_path):
        print(full_path)


하위폴더 목록:
../Projects/OSU_LWF/DigitalTwin/Test/cp
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_001
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_002
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_003
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_004
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_005
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_006
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_007
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_008
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_009
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_010
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_011
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_012
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_013
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_014
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_015
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_016
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_017
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_018
../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_019
../Projects/OSU_LWF/DigitalTwin/Test/cy

In [ ]:
# ---- 모든 하위 폴더 반복 처리 ----

subfolders = [
    os.path.join(forest_base, d)
    for d in os.listdir(forest_base)
    if os.path.isdir(os.path.join(forest_base, d))
]
if not subfolders:
    raise RuntimeError(f"Forest_v2 디렉토리 내에 하위폴더가 없습니다: {forest_base}")

for folder_path in sorted(subfolders):
    folder_basename = os.path.basename(folder_path.rstrip('/'))
    save_excel_path = os.path.join(save_folder, f'merged_Targets_{folder_basename}.xlsx')
    merge_targets_in_folder(folder_path, save_excel_path)


[DONE] ../Projects/OSU_LWF/DigitalTwin/Test/cp → ./Test_results/merged_Targets_cp.xlsx
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_001: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_002: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_003: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_004: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_005: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_006: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_007: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_008: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_009: gridTarget/particleTarget csv 파일 없음
[SKIP] ../Projects/OSU_LWF/DigitalTwin/Test/cyl_in_010: gridTarget/particleTarget csv